In [ ]:
# ==============================================================================
# CODE HOÀN CHỈNH: 6 CHỈ SỐ KHÍ HẬU ĐBSCL 2025 - ML READY (NO NODATA FOR RAIN)
# ==============================================================================

# =========================
# BƯỚC 0: CÀI ĐẶT
# =========================
!pip install geemap -q

import ee
import geemap
import os
from google.colab import drive

# =========================
# BƯỚC 1: KẾT NỐI
# =========================
print("📂 Mount Google Drive...")
drive.mount('/content/drive')

YEAR = 2025
out_dir = f'/content/drive/MyDrive/Du_Lieu_DBSH_{YEAR}_ML_READY'
os.makedirs(out_dir, exist_ok=True)
print(f"✅ Output dir: {out_dir}")

PROJECT_ID = 'geemap-mekong-483717'

try:
    ee.Initialize(project=PROJECT_ID)
except:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

print("✅ GEE connected")

# =========================
# BƯỚC 2: ROI & THỜI GIAN
# =========================
redriver_provinces = [
    'Ha Noi City', 'Hai Phong City', 'Quang Ninh', 'Bac Ninh', 'Bac Giang',
    'Hai Duong', 'Hung Yen', 'Thai Binh',
    'Nam Dinh', 'Ha Nam', 'Ninh Binh'
]

vietnam = ee.FeatureCollection("FAO/GAUL/2015/level1")
roi = vietnam.filter(
    ee.Filter.inList('ADM1_NAME', redriver_provinces)
).geometry()

months_info = [
    {'num': 1, 'name': '01', 'days': 31},
    {'num': 2, 'name': '02', 'days': 28},
    {'num': 3, 'name': '03', 'days': 31},
    {'num': 4, 'name': '04', 'days': 30},
    {'num': 5, 'name': '05', 'days': 31},
    {'num': 6, 'name': '06', 'days': 30},
    {'num': 7, 'name': '07', 'days': 31},
    {'num': 8, 'name': '08', 'days': 31},
    {'num': 9, 'name': '09', 'days': 30},
    {'num': 10, 'name': '10', 'days': 31},
    {'num': 11, 'name': '11', 'days': 30},
    {'num': 12, 'name': '12', 'days': 31}
]

# =========================
# BƯỚC 3: VÒNG LẶP XỬ LÝ
# =========================
for month in months_info:
    mnum = month['num']
    mname = month['name']

    ee_start = ee.Date(f'{YEAR}-{mnum:02d}-01')
    ee_end = ee_start.advance(1, 'month')

    print(f"\n📆 Processing {mname}/{YEAR}")

    # ------------------------------------------------------------------
    # 1. RAIN (CHIRPS) - NO NODATA
    # ------------------------------------------------------------------
    print("   🌧️ Rain (CHIRPS)")
    chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY") \
        .filterDate(ee_start, ee_end) \
        .select('precipitation')

    def get_rain(d):
        img = chirps.filterDate(
            ee.Date(d),
            ee.Date(d).advance(1, 'day')
        ).sum()

        return img.unmask(0) \
            .clip(roi) \
            .rename('Rain_mm') \
            .set('system:time_start', ee.Date(d).millis()) \
            .float()

    days = ee.List.sequence(0, month['days'] - 1)
    dates = days.map(lambda n: ee_start.advance(n, 'day'))
    rain_col = ee.ImageCollection(dates.map(get_rain))

    geemap.ee_export_image(
        rain_col.toBands(),
        os.path.join(out_dir, f'1_Rain_{YEAR}_{mname}.tif'),
        scale=5000,
        region=roi,
        file_per_band=False
    )

    # ------------------------------------------------------------------
    # 2. TAVG
    # ------------------------------------------------------------------
    print("   🌡️ Tavg")
    tavg = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR") \
        .filterDate(ee_start, ee_end) \
        .select('temperature_2m') \
        .map(lambda img: img.subtract(273.15)
             .clip(roi).rename('Tavg').float())

    geemap.ee_export_image(
        tavg.toBands(),
        os.path.join(out_dir, f'2_Tavg_{YEAR}_{mname}.tif'),
        scale=10000,
        region=roi,
        file_per_band=False
    )

    # ------------------------------------------------------------------
    # 3. RH
    # ------------------------------------------------------------------
    print("   💧 RH")
    era5_rh = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR") \
        .filterDate(ee_start, ee_end) \
        .select(['temperature_2m', 'dewpoint_temperature_2m'])

    def calc_rh(img):
        T = img.select('temperature_2m').subtract(273.15)
        Td = img.select('dewpoint_temperature_2m').subtract(273.15)
        es = T.expression(
            '6.112 * exp((17.67*T)/(T+243.5))', {'T': T}
        )
        e = Td.expression(
            '6.112 * exp((17.67*Td)/(Td+243.5))', {'Td': Td}
        )
        return e.divide(es).multiply(100).clamp(0, 100) \
            .rename('RH').clip(roi).float()

    rh = era5_rh.map(calc_rh)

    geemap.ee_export_image(
        rh.toBands(),
        os.path.join(out_dir, f'3_RH_{YEAR}_{mname}.tif'),
        scale=10000,
        region=roi,
        file_per_band=False
    )

    # ------------------------------------------------------------------
    # 4. SOLAR
    # ------------------------------------------------------------------
    print("   ☀️ Solar")
    solar = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR") \
        .filterDate(ee_start, ee_end) \
        .select('surface_solar_radiation_downwards_sum') \
        .map(lambda img: img.divide(1e6)
             .clip(roi).rename('Solar_MJ').float())

    geemap.ee_export_image(
        solar.toBands(),
        os.path.join(out_dir, f'4_Solar_{YEAR}_{mname}.tif'),
        scale=10000,
        region=roi,
        file_per_band=False
    )

    # ------------------------------------------------------------------
    # 5. TMAX
    # ------------------------------------------------------------------
    print("   🔥 Tmax")
    tmax = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR") \
        .filterDate(ee_start, ee_end) \
        .select('temperature_2m_max') \
        .map(lambda img: img.subtract(273.15)
             .clip(roi).rename('Tmax').float())

    geemap.ee_export_image(
        tmax.toBands(),
        os.path.join(out_dir, f'5_Tmax_{YEAR}_{mname}.tif'),
        scale=10000,
        region=roi,
        file_per_band=False
    )

    # ------------------------------------------------------------------
    # 6. TMIN
    # ------------------------------------------------------------------
    print("   ❄️ Tmin")
    tmin = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR") \
        .filterDate(ee_start, ee_end) \
        .select('temperature_2m_min') \
        .map(lambda img: img.subtract(273.15)
             .clip(roi).rename('Tmin').float())

    geemap.ee_export_image(
        tmin.toBands(),
        os.path.join(out_dir, f'6_Tmin_{YEAR}_{mname}.tif'),
        scale=10000,
        region=roi,
        file_per_band=False
    )

print("\n✅ DONE – 2025 ML READY DATASET CREATED")
